# Inference and Visualization

Run trained change detection models on new data and visualize results.
This notebook covers single-pair inference, sliding window for large scenes,
and batch evaluation with quality assessment.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.config import Config
from src.data_loader import build_dataloaders, load_image, load_mask
from src.eval import load_model_from_checkpoint, predict_single, sliding_window_inference, evaluate_test_set
from src.utils import (
    get_device, set_seed, compute_metrics,
    visualize_change_detection, normalize_for_display,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load config and model
config = Config.from_yaml('../configs/default.yaml')
device = get_device()
set_seed(config.seed)

CHECKPOINT_PATH = Path('../models/checkpoints/best_model.pth')

if CHECKPOINT_PATH.exists():
    model = load_model_from_checkpoint(CHECKPOINT_PATH, config, device)
    print('Model loaded successfully.')
else:
    print(f'Checkpoint not found at {CHECKPOINT_PATH}')
    print('Train a model first: python -m src.train --config configs/default.yaml')

## 1. Single Pair Inference

Load a single pre/post image pair and run change detection.
This is the simplest inference mode for quick checks.

In [ ]:
# Load a single image pair from the test set
test_dir = Path('../data/raw/test')

if test_dir.exists():
    import os
    filenames = sorted(os.listdir(test_dir / 'A'))
    sample_name = filenames[0]
    
    # Load images (these come as H, W, C numpy arrays)
    img1_raw = load_image(test_dir / 'A' / sample_name)
    img2_raw = load_image(test_dir / 'B' / sample_name)
    gt_mask = load_mask(test_dir / 'label' / sample_name)
    
    print(f'Image shape: {img1_raw.shape}')
    print(f'Mask shape: {gt_mask.shape}')
    print(f'Change fraction: {gt_mask.mean():.4f}')
    
    # Preprocess for model (normalize + to tensor)
    from src.data_loader import get_transforms
    transform = get_transforms('test', patch_size=256)
    transformed = transform(image=img1_raw, image2=img2_raw, mask=gt_mask)
    
    img1_tensor = transformed['image']
    img2_tensor = transformed['image2']
    
    # Run inference
    pred_mask = predict_single(model, img1_tensor, img2_tensor, device, threshold=0.5)
    
    # Compute metrics
    metrics = compute_metrics(pred_mask, gt_mask[:256, :256].astype(int))
    print(f'\\nMetrics: {metrics}')
    
    # Visualize
    fig = visualize_change_detection(
        image_t1=img1_tensor.numpy(),
        image_t2=img2_tensor.numpy(),
        prediction=pred_mask,
        ground_truth=gt_mask[:256, :256],
        title=f'{sample_name} | F1: {metrics["f1"]:.3f}',
    )
    plt.show()
else:
    print(f'Test data not found at {test_dir}')

## 2. Sliding Window Inference

For images larger than the training patch size (256x256), we use a sliding
window approach. The window moves across the image with overlap, and
predictions in overlapping regions are averaged for smoother results.

In [ ]:
# Demonstrate sliding window inference on a larger image
# This handles the case where test images are larger than 256x256

if test_dir.exists() and CHECKPOINT_PATH.exists():
    # Use the raw image (before cropping)
    img1_raw_t = torch.from_numpy(img1_raw.transpose(2, 0, 1)).float() / 255.0
    img2_raw_t = torch.from_numpy(img2_raw.transpose(2, 0, 1)).float() / 255.0
    
    # Normalize with ImageNet stats
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img1_norm = (img1_raw_t - mean) / std
    img2_norm = (img2_raw_t - mean) / std
    
    print(f'Full image shape: {img1_norm.shape}')
    
    # Run sliding window with 50% overlap
    pred_full = sliding_window_inference(
        model, img1_norm, img2_norm, device,
        patch_size=256, stride=128, threshold=0.5,
    )
    
    print(f'Prediction shape: {pred_full.shape}')
    print(f'Predicted change fraction: {pred_full.mean():.4f}')
    
    # Display
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(normalize_for_display(img1_norm.numpy()))
    axes[0].set_title('Pre-Change')
    axes[1].imshow(normalize_for_display(img2_norm.numpy()))
    axes[1].set_title('Post-Change')
    axes[2].imshow(pred_full, cmap='hot')
    axes[2].set_title('Predicted Changes')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Model or test data not available.')

## 3. Batch Test Set Evaluation

Run the full evaluation pipeline on the test set and report metrics.

In [ ]:
if CHECKPOINT_PATH.exists():
    loaders = build_dataloaders(
        data_root=config.paths.data_root,
        batch_size=config.training.batch_size,
        num_workers=0,
    )
    
    if 'test' in loaders:
        metrics = evaluate_test_set(
            model, loaders['test'], device, config,
            save_visualizations=True, max_vis=10,
        )
        
        print('\\nTest Results:')
        for k, v in metrics.items():
            print(f'  {k}: {v:.4f}')
    else:
        print('Test split not found.')
else:
    print('No checkpoint available.')

## 4. Prediction Gallery

Sort predictions by F1 score to find the best and worst performing samples.
This helps identify failure modes (e.g., cloud shadows, seasonal changes).

In [ ]:
def prediction_gallery(model, loader, device, config, n_best=3, n_worst=3):
    """Generate gallery of best and worst predictions by F1 score."""
    model.eval()
    results = []
    
    for batch in loader:
        img1 = batch['image1'].to(device)
        img2 = batch['image2'].to(device)
        
        with torch.no_grad():
            output = model(img1, img2)
        
        preds = (torch.sigmoid(output['pred']) > 0.5).cpu().numpy()
        
        for i in range(img1.shape[0]):
            pred = preds[i, 0]
            gt = batch['mask'][i, 0].numpy()
            m = compute_metrics(pred, gt)
            results.append({
                'f1': m['f1'],
                'img1': batch['image1'][i].numpy(),
                'img2': batch['image2'][i].numpy(),
                'pred': pred,
                'gt': gt,
                'name': batch['filename'][i],
            })
    
    # Sort by F1
    results.sort(key=lambda x: x['f1'], reverse=True)
    
    print(f'=== Best {n_best} Predictions ===')
    for r in results[:n_best]:
        fig = visualize_change_detection(
            r['img1'], r['img2'], r['pred'], r['gt'],
            title=f"{r['name']} | F1: {r['f1']:.3f}"
        )
        plt.show()
    
    # Only show worst if there are enough with actual changes
    changed = [r for r in results if r['gt'].sum() > 100]
    if changed:
        changed.sort(key=lambda x: x['f1'])
        print(f'\\n=== Worst {n_worst} Predictions (with changes) ===')
        for r in changed[:n_worst]:
            fig = visualize_change_detection(
                r['img1'], r['img2'], r['pred'], r['gt'],
                title=f"{r['name']} | F1: {r['f1']:.3f}"
            )
            plt.show()

# Uncomment when model and data are available:
# prediction_gallery(model, loaders['test'], device, config)

## 5. Export Predictions as GeoTIFF

Save predictions in geospatial format so they can be loaded in GIS
software (QGIS, ArcGIS) and overlaid on maps.

In [ ]:
def save_prediction_geotiff(
    prediction: np.ndarray,
    reference_path: str,
    output_path: str,
):
    """Save a binary change mask as a GeoTIFF, copying geospatial
    metadata (CRS, transform) from the source image.
    
    This allows the prediction to be opened in GIS software with
    correct geographic coordinates.
    
    Args:
        prediction: Binary change mask (H, W).
        reference_path: Path to source GeoTIFF (for CRS and transform).
        output_path: Where to save the prediction GeoTIFF.
    """
    try:
        import rasterio
        from rasterio.transform import from_bounds
    except ImportError:
        print('rasterio not installed. Saving as PNG instead.')
        from PIL import Image
        Image.fromarray((prediction * 255).astype(np.uint8)).save(
            output_path.replace('.tif', '.png')
        )
        return
    
    # Read geospatial metadata from reference image
    with rasterio.open(reference_path) as src:
        profile = src.profile.copy()
    
    # Update profile for single-band binary output
    profile.update(
        dtype='uint8',
        count=1,
        compress='lzw',
    )
    
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(prediction.astype(np.uint8), 1)
    
    print(f'Saved prediction to {output_path}')

# Example usage:
# save_prediction_geotiff(
#     prediction=pred_full,
#     reference_path='../data/raw/test/A/scene_001.tif',
#     output_path='../results/predictions/scene_001_change.tif',
# )
print('GeoTIFF export function ready.')